## Load data from hf

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
from datasets import load_dataset

In [ ]:
df = load_dataset("viamr-project/reasoning-frame-hint", token=os.getenv("HF_TOKEN"))
df

In [ ]:
sample = df["train"][0]

# vLLM

In [ ]:
from vllm import LLM, SamplingParams

In [ ]:
import os
from vllm import SamplingParams
# old: viamr-project/qwen3-1.7b-amr-20260704-0113
# new: viamr-project/qwen3-1.7b-amr-20260705-0434
vllm_config = {
    "model": "viamr-project/qwen3-1.7b-amr-20260704-0113",
    "tensor_parallel_size": 1,
    "gpu_memory_utilization": 0.85,
    "max_model_len": 4096,
    
    "temperature": 0.60,
    "top_p": 0.95,
    "top_k": 20,
    "min_p": 0.0,
    "presence_penalty": 1.5,
    "repetition_penalty": 1.0,
    "max_tokens": 2048,
    "enable_thinking": True,
}

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

sampling_params = SamplingParams(
    temperature=vllm_config["temperature"],
    top_p=vllm_config["top_p"],
    top_k=vllm_config["top_k"],
    min_p=vllm_config["min_p"],
    presence_penalty=vllm_config["presence_penalty"],
    repetition_penalty=vllm_config["repetition_penalty"],
    max_tokens=vllm_config["max_tokens"],
)

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA device count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"Current CUDA device: {torch.cuda.current_device()}")
    print(f"Device name: {torch.cuda.get_device_name()}")


try:
    vllm_model = LLM(
        model=vllm_config["model"],
        tensor_parallel_size=vllm_config["tensor_parallel_size"],
        gpu_memory_utilization=vllm_config["gpu_memory_utilization"],
        max_model_len=vllm_config["max_model_len"],
        hf_token=os.getenv("HF_TOKEN")
    )
    print(f"✓ vLLM model loaded successfully: {vllm_config["model"]}")
except Exception as e:
    print(f"❌ Error loading model: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
from pyexpat.errors import messages

from vllm import LLM

prompt = "Convert the following English sentence into its Abstract Meaning Representation (AMR):\n\n<sentence>The Cat sit on the mat</sentence>"


messages = [
    {"role": "user", "content": prompt}
]
outputs = vllm_model.chat(
            messages,
            sampling_params=sampling_params,
            chat_template_kwargs={"enable_thinking": vllm_config["enable_thinking"]},
        )

# outputs = vllm_model.chat(prompt, sampling_params)
for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"Prompt: {prompt!r}, Generated text: {generated_text!r}")

In [ ]:
print(generated_text)

## Đánh giá độ chính xác với smatchpp

In [ ]:
import json
import re
from tqdm import tqdm
from smatchpp import Smatchpp

# ── helpers ────────────────────────────────────────────────────────────────────

def fix_amr_parentheses(amr_str: str) -> str:
    """
    Balances parentheses in an AMR string.
    Appends missing closing parentheses or removes extra ones from the end.
    """
    if not amr_str:
        return amr_str
        
    open_count = amr_str.count('(')
    close_count = amr_str.count(')')
    
    if open_count > close_count:
        amr_str += ')' * (open_count - close_count)
    elif close_count > open_count:
        diff = close_count - open_count
        for _ in range(diff):
            idx = amr_str.rfind(')')
            if idx != -1:
                amr_str = amr_str[:idx] + amr_str[idx+1:]
                
    return amr_str

def extract_amr_from_response(model_response: list[str] | str) -> str | None:
    """
    Extract AMR string from <amr>...</amr> tags in model_response.
    model_response can be a list of strings or a plain string.
    Returns the first match found, or None.
    """
    text = "\n".join(model_response) if isinstance(model_response, list) else model_response
    match = re.search(r"<amr>(.*?)</amr>", text, re.DOTALL)
    if match:
        amr_str = match.group(1).strip()
        return fix_amr_parentheses(amr_str)
    return None


def score_amr_pair(gold: str, pred: str, scorer: Smatchpp) -> dict:
    """
    Score a single (gold, pred) AMR pair.
    Returns {'F1': float, 'Precision': float, 'Recall': float}
    or {'F1': 0.0, 'Precision': 0.0, 'Recall': 0.0} on parse errors.
    """
    try:
        result = scorer.score_pair(gold, pred)
        return result["main"]          # {'F1': ..., 'Precision': ..., 'Recall': ...}
    except Exception as e:
        return {"F1": 0.0, "Precision": 0.0, "Recall": 0.0, "error": str(e)}


# ── main ───────────────────────────────────────────────────────────────────────

def evaluate_hf_dataset(dataset, batch_size: int = 128) -> None:
    """
    Evaluate the model on a Hugging Face dataset (e.g., df['test']).
    Generates AMR predictions using the global `vllm_model` and `sampling_params`,
    then calculates Smatch score against the gold AMR.
    """
    scorer = Smatchpp()
    results = []
    skipped = 0

    print(f"Preparing messages for {len(dataset)} samples...")
    # Prepare all prompts
    batch_messages = []
    for item in dataset:
        sentence = item.get("sentence", "")
        prompt = (
            "Convert the following English sentence into its Abstract Meaning Representation (AMR):\n\n"
            f"<sentence>{sentence}</sentence>"
        )
        batch_messages.append([{"role": "user", "content": prompt}])

    print("Generating model responses via vLLM...")
    # Generate in batches
    model_responses = []
    for i in tqdm(range(0, len(batch_messages), batch_size)):
        batch = batch_messages[i:i+batch_size]
        outputs = vllm_model.chat(
            batch,
            sampling_params=sampling_params,
            chat_template_kwargs={"enable_thinking": vllm_config["enable_thinking"]},
            use_tqdm=False
        )
        for output in outputs:
            model_responses.append(output.outputs[0].text)

    print("Scoring predictions...")
    for i, item in enumerate(dataset):
        gold_amr = item.get("amr", "").strip()
        sentence = item.get("sentence", "")
        model_response = model_responses[i]

        pred_amr = extract_amr_from_response(model_response)

        if not gold_amr or not pred_amr:
            print(f"[{i}] SKIP — missing {'gold' if not gold_amr else 'pred'} AMR | sentence: {sentence!r}")
            skipped += 1
            continue

        scores = score_amr_pair(gold_amr, pred_amr, scorer)

        results.append({
            "index":     i,
            "sentence":  sentence,
            "gold_amr":  gold_amr,
            "pred_amr":  pred_amr,
            "model_response": model_response,
            **scores,
        })

        if i % 100 == 0 or i == len(dataset) - 1:
            print(
                f"[{i:>4}/{len(dataset)}] F1={scores['F1']:5.1f}  "
                f"P={scores['Precision']:5.1f}  "
                f"R={scores['Recall']:5.1f}  | {sentence[:70]}"
            )

    # ── aggregate ──────────────────────────────────────────────────────────────
    if results:
        avg_f1   = sum(r["F1"]        for r in results) / len(results)
        avg_prec = sum(r["Precision"] for r in results) / len(results)
        avg_rec  = sum(r["Recall"]    for r in results) / len(results)

        print("\n" + "=" * 70)
        print(f"  Evaluated : {len(results)} samples  (skipped: {skipped})")
        print(f"  Avg F1    : {avg_f1:.2f}")
        print(f"  Avg Prec  : {avg_prec:.2f}")
        print(f"  Avg Recall: {avg_rec:.2f}")
        print("=" * 70)
    else:
        print("No valid pairs to evaluate.")

    # ── save detailed results ──────────────────────────────────────────────────
    out_path = "amr_eval_results.jsonl"
    with open(out_path, "w", encoding="utf-8") as f:
        for r in results:
            # convert numpy floats → plain Python floats for JSON
            row = {k: (float(v) if hasattr(v, "item") else v) for k, v in r.items()}
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"\nDetailed results saved to: {out_path}")


if __name__ == "__main__":
    # By default, use the test set from the loaded HF dataset 'df'
    if 'df' in globals():
        evaluate_hf_dataset(df["test"])
    else:
        print("Please load the dataset 'df' first.")
